# FPL Season Manager Test

Run this notebook to simulate a stateful FPL manager using the prediction CSV files in `outputs/`.

The manager keeps season state: initial squad, free transfers, transfer hits, bank, chips, captain, vice-captain, bench value, and decision history.

In [ ]:
from pathlib import Path

import pandas as pd

from fplmodel.season_manager import (
    SeasonManagerConfig,
    load_prediction_files,
    run_repeated_season_simulations,
    save_season_simulation_artifact,
)

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 80)

## Settings

Adjust these values before running. You do not choose players, transfers, captaincy, vice-captaincy, or chips manually; the AI manager does that inside the simulation.

`SIMULATION_MODE = "periodic_reoptimization"` means the AI manager re-chooses its own squad, transfers, captains, vice-captains, and chips every `POLICY_REFRESH_INTERVAL` simulations. This is the recommended mode for 50,000 simulations.

`SIMULATION_MODE = "fixed_policy"` means the AI manager first chooses its own squad, transfers, captains, vice-captains, and chips from the expected projections, then runs many stochastic point simulations against that AI-chosen plan. This is the fastest mode for 50,000 simulations.

`SIMULATION_MODE = "full_reoptimization"` means the AI manager re-picks squad moves, captains, vice-captains, and chips separately inside every simulation from sampled projections. This is much slower because it re-solves the manager season thousands of times.

In [ ]:
OUTPUT_DIR = Path("outputs")
START_GW = 1  # set to 1 for a full future season once GW1 predictions exist
END_GW = 38      # keep this at 38 to avoid loading any post-season GW39 artifact
SIMULATIONS = 50000
SIMULATION_MODE = "periodic_reoptimization"  # use "full_reoptimization" only for small batches
POLICY_REFRESH_INTERVAL = 1000
RANDOM_SEED = 90
SEASON_LABEL = "simulation season 2025-2026"
SHOW_PROGRESS = True
PROGRESS_GAMEWEEK_INTERVAL = 10

config = SeasonManagerConfig(
    initial_horizon=4,
    transfer_horizon=4,
    chip_lookahead=4,
    transfer_gain_threshold=1.5,
    max_transfers_per_gw=2,
)

## Load Prediction Files

In [ ]:
predictions = load_prediction_files(OUTPUT_DIR, start_gw=START_GW, end_gw=END_GW)
available_gws = sorted(predictions)

expected_gws = list(range(START_GW or 1, END_GW or max(available_gws) + 1))
missing_gws = [gw for gw in expected_gws if gw not in available_gws]

print(f"Loaded {len(available_gws)} gameweeks: GW{available_gws[0]} to GW{available_gws[-1]}")
if missing_gws:
    print(f"Missing prediction files for: {missing_gws}")
pd.DataFrame(
    {
        "gameweek": available_gws,
        "players": [len(predictions[gw]) for gw in available_gws],
        "top_projected_player": [
            predictions[gw].sort_values("expected_points", ascending=False).iloc[0]["full_name"]
            for gw in available_gws
        ],
        "top_expected_points": [
            predictions[gw].sort_values("expected_points", ascending=False).iloc[0]["expected_points"]
            for gw in available_gws
        ],
    }
)

## Run Repeated Stateful Simulations

In [ ]:
result = run_repeated_season_simulations(
    predictions,
    simulations=SIMULATIONS,
    config=config,
    random_seed=RANDOM_SEED,
    show_progress=SHOW_PROGRESS,
    season_label=SEASON_LABEL,
    progress_gameweek_interval=PROGRESS_GAMEWEEK_INTERVAL,
    simulation_mode=SIMULATION_MODE,
    policy_refresh_interval=POLICY_REFRESH_INTERVAL,
)

summary = pd.DataFrame([result["summary"]])
summary

## Compare Simulation Runs

In [ ]:
runs_df = pd.DataFrame(
    [
        {
            "simulation_id": run["simulation_id"],
            "total_expected_points": run["summary"]["total_expected_points"],
            "transfers_made": run["summary"]["transfers_made"],
            "chips_used": ", ".join(run["summary"].get("chips_used", [])),
        }
        for run in result["runs"]
    ]
).sort_values("total_expected_points", ascending=False)

runs_df.head(20)

## Best Run: Weekly Decisions

In [ ]:
best_run = result["best_run"]

decision_rows = []
for decision in best_run["decisions"]:
    decision_rows.append(
        {
            "GW": decision["gameweek"],
            "expected_points": round(decision["expected_points"], 2),
            "before_hits": round(decision["expected_points_before_transfer_hits"], 2),
            "hit_cost": round(decision["transfer_hit_cost"], 2),
            "captain": decision["captain"],
            "vice_captain": decision["vice_captain"],
            "chip": decision["chip"],
            "chip_gain": round(decision["chip_gain"], 2),
            "transfers": len(decision["transfers"]),
            "transfer_gain": round(decision["transfer_gain"], 2),
            "free_transfers_before": decision["free_transfers_before"],
            "free_transfers_after": decision["free_transfers_after"],
            "bank_m": decision["bank_m"],
            "team_value_m": decision["team_value_m"],
            "squad_sale_value_m": decision["squad_sale_value_m"],
            "bench_expected_points": round(decision["bench_expected_points"], 2),
        }
    )

decisions_df = pd.DataFrame(decision_rows)
decisions_df

## Best Run: Transfers

In [ ]:
transfer_rows = []
for decision in best_run["decisions"]:
    for transfer in decision["transfers"]:
        transfer_rows.append(
            {
                "GW": decision["gameweek"],
                "out": transfer["out_player"]["full_name"],
                "out_team": transfer["out_player"]["team_name"],
                "out_current_price": transfer["out_player"]["now_cost_millions"],
                "out_purchase_price": transfer["out_purchase_price"],
                "out_sale_value": transfer["out_sale_value"],
                "in": transfer["in_player"]["full_name"],
                "in_team": transfer["in_player"]["team_name"],
                "in_purchase_price": transfer["in_purchase_price"],
                "expected_points_delta": round(transfer["expected_points_delta"], 2),
            }
        )

transfers_df = pd.DataFrame(transfer_rows)
transfers_df if not transfers_df.empty else pd.DataFrame({"message": ["No transfers made in best run"]})

## Best Run: Chips

In [ ]:
chips_df = decisions_df[decisions_df["chip"].notna() & (decisions_df["chip"] != "")][
    ["GW", "chip", "chip_gain", "captain", "expected_points", "bench_expected_points"]
]
chips_df if not chips_df.empty else pd.DataFrame({"message": ["No chips used in best run"]})

## First Gameweek Squad

In [ ]:
first_decision = best_run["decisions"][0]
first_squad_rows = []

for player in first_decision["team"]["squad"]:
    first_squad_rows.append({**player, "role": "starter"})
for player in first_decision["team"]["bench"]:
    first_squad_rows.append({**player, "role": "bench"})

first_squad_df = pd.DataFrame(first_squad_rows)
display_cols = [
    "role",
    "full_name",
    "team_name",
    "element_type",
    "now_cost_millions",
    "expected_points",
    "captain",
    "vice_captain",
    "bench_order",
]
first_squad_df[[col for col in display_cols if col in first_squad_df.columns]]

## Save Results

In [ ]:
artifact_path = save_season_simulation_artifact(
    result,
    OUTPUT_DIR / "season_plan_simulations.json",
)

print(f"Saved full simulation artifact to {artifact_path}")